# 🚀 R2AI Stage 3 2026: Fast Cross-Encoder Reranking on Kaggle GPU

Pipeline 2-Stage Retrieval:
1. **Input:** `candidates_stage1_chunks250.parquet` (Được sinh từ Local FAISS Hybrid 1M Corpus, giữ nhiều chunks / doc).
2. **Model:** `BAAI/bge-reranker-v2-m3` (FP16, Batch size 128).
3. **Score Blending:** Dung hợp điểm tương quan sâu sắc $0.75 \times \text{Reranker} + 0.25 \times \text{Stage 1}$ để vừa bứt phá Precision, vừa bảo toàn đỉnh cao Docs Recall và Chunks Recall.
4. **Output:** `submission_bgem3_cross_encoder_reranked.zip` (< 100 MB, tuân thủ 100% định dạng BTC).

In [ ]:
# 1. Cài đặt FlagEmbedding
!pip install -q FlagEmbedding

In [ ]:
import os
import gc
import json
import zipfile
import time
import numpy as np
import pandas as pd
from FlagEmbedding import FlagReranker

# 2. Tìm tệp candidate đầu vào
candidate_file = None
for root, _, files in os.walk("/kaggle"):
    for f in files:
        if f.endswith(".parquet") and "candidate" in f.lower():
            candidate_file = os.path.join(root, f)
            break
    if candidate_file:
        break

if not candidate_file:
    for fallback in ["candidates_stage1_chunks250.parquet", "candidates_stage1_top150.parquet"]:
        if os.path.exists(fallback):
            candidate_file = fallback
            break

print(f"📂 File ứng viên: {candidate_file}")
df = pd.read_parquet(candidate_file)
print(f"Đã nạp {len(df):,} cặp ứng viên trên {df['query_id'].nunique():,} câu hỏi.")

In [ ]:
# 3. Ghép cặp câu hỏi - đoạn văn đa ngữ chuẩn xác
print("Forming cross-lingual pairs...")
pairs = []
for row in df.itertuples(index=False):
    chunk_text = str(row.chunk_text or "")
    
    # Phát hiện chữ Hán để so khớp với query tiếng Trung
    is_zh = any('\u4e00' <= char <= '\u9fff' for char in chunk_text[:200])
    # Phát hiện văn bản tiếng Anh (PubMed)
    ascii_chars = sum(1 for c in chunk_text[:200] if ord(c) < 128 and c.isalpha())
    is_en = (ascii_chars / max(len(chunk_text[:200]), 1)) > 0.75 and not any(ord(c) > 255 for c in chunk_text[:100])

    if is_zh and hasattr(row, "query_zh") and row.query_zh and str(row.query_zh).strip():
        q_text = str(row.query_zh)
    elif is_en and hasattr(row, "query_en") and row.query_en and str(row.query_en).strip():
        q_text = str(row.query_en)
    else:
        q_text = str(row.query_vi)

    pairs.append([q_text, chunk_text])

print(f"Tổng số cặp: {len(pairs):,}")

In [ ]:
# 4. Khởi tạo mô hình BGE-Reranker-v2-m3 trên GPU
print("🚀 Loading BAAI/bge-reranker-v2-m3 on GPU...")
reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True)

# 5. Chấm điểm siêu tốc với Batch Size 128 trên Kaggle T4/P100
batch_size = 128
max_length = 512
t_score = time.time()
print(f"⚡ Bắt đầu chấm điểm {len(pairs):,} cặp...")

scores = reranker.compute_score(pairs, batch_size=batch_size, max_length=max_length)
if isinstance(scores, (int, float)):
    scores = [float(scores)]

df["reranker_score"] = scores
print(f"✅ Hoàn tất trong {time.time() - t_score:.2f}s (Tốc độ: {len(pairs)/(time.time() - t_score):.1f} pairs/sec)!")

del reranker, pairs
gc.collect()

In [ ]:
# 6. Dung hợp điểm số thông minh (Score Blending) & Trích xuất Top 100 Docs & Top 98 Chunks
def normalize_group(g):
    s1 = g["stage1_score"].values
    s2 = g["reranker_score"].values
    s1_norm = (s1 - s1.min()) / (s1.max() - s1.min() + 1e-6) if s1.max() > s1.min() else np.ones_like(s1)
    s2_norm = (s2 - s2.min()) / (s2.max() - s2.min() + 1e-6) if s2.max() > s2.min() else np.ones_like(s2)
    # 75% Cross-Encoder + 25% Stage 1 Hybrid
    g["final_score"] = 0.75 * s2_norm + 0.25 * s1_norm
    return g

print("Dung hợp điểm số và lọc ứng viên Top 100 Docs & Top 98 Chunks...")
df = df.groupby("query_id", group_keys=False).apply(normalize_group)

submission_data = []
for q_id, group in df.groupby("query_id"):
    sorted_g = group.sort_values("final_score", ascending=False)
    seen_docs = set()
    seen_chunks = set()
    rel_docs = []
    rel_chunks = []

    for row in sorted_g.itertuples(index=False):
        doc_id = int(row.doc_id)
        chunk_txt = str(row.chunk_text or "").strip()

        # Top 100 unique documents (theo thứ tự chunk điểm cao nhất của document đó)
        if doc_id not in seen_docs and len(rel_docs) < 100:
            seen_docs.add(doc_id)
            rel_docs.append(doc_id)

        # Top 98 chunks (tự động cho phép document đúng đóng góp nhiều chunks!)
        if chunk_txt and chunk_txt not in seen_chunks and len(rel_chunks) < 98:
            seen_chunks.add(chunk_txt)
            rel_chunks.append({"doc_id": doc_id, "chunk_text": chunk_txt})

    submission_data.append({
        "id": int(q_id),
        "relevant_docs": rel_docs,
        "relevant_chunks": rel_chunks
    })

submission_data.sort(key=lambda x: x["id"])

# 7. Xuất file submission.json & nén submission.zip
out_json = "submission_bgem3_cross_encoder_reranked.json"
out_zip = "submission_bgem3_cross_encoder_reranked.zip"

with open(out_json, "w", encoding="utf-8") as f:
    json.dump(submission_data, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(out_json, arcname="submission.json")

sz_mb = os.path.getsize(out_zip) / (1024 * 1024)
print("=" * 60)
print(f"🎉 THÀNH CÔNG! File zip: {out_zip} ({sz_mb:.2f} MB)")
print(f"Tổng queries: {len(submission_data)} | An toàn < 100MB: {sz_mb < 100}")
print("=" * 60)